In [1]:
import os
import pandas as pd
import numpy as np
import xarray as xr
import dask
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from dask import delayed

# ============================
# User settings
# ============================
sdate, edate = '20090701', '20240630'
write_path = '/scratch/ng72/ms5578/solar_wind_tseries/'

# BARRA-R2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")
cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")

# Dask cluster

client = Client(n_workers=12,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45445,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40207,Total threads: 2
Dashboard: /proxy/40385/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:43059,


In [2]:
boco = cluster_dates

In [3]:
def get_days(days, nc_dir):
    date_list = pd.to_datetime(days['date'])

    # Build filename filter
    all_files = os.listdir(nc_dir)
    selected_files = [
        os.path.join(nc_dir, f)
        for f in all_files
        if any(d.strftime("%Y%m") in f for d in date_list)
    ]

    # Open multiple files lazily with parallel reads
    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks='auto'
    )

    # Select all hours of the requested dates
    subset = ds.where(ds.time.dt.floor('D').isin(date_list), drop=True)

    return subset


In [4]:
u_boco = get_days(boco, u_path)
v_boco = get_days(boco, v_path)

boco = xr.merge([u_boco, v_boco])
boco = boco.chunk({"time": 168, "lat": -1, "lon": -1})

In [5]:
# Write to NetCDF using compute=False
delayed_obj = boco.to_netcdf(
                "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc",
                engine="netcdf4",
                mode="w",
                unlimited_dims=["time"],   # allows appending along time dimension
                compute=False
                )

# Trigger computation with Dask
with ProgressBar():
    dask.compute(delayed_obj)